# 01 — Generate Tool-Calling Training Data (extensive)

**CPU only. Run in Modal notebook before notebook 02.**

Teaches Moshi, via its inner-monologue text stream, to:
1. Emit `<|tool_call|>get_time<|tool_end|>` when the user asks for the time
2. Emit `<|tool_call|>get_weather CITY<|tool_end|>` for weather
3. Read the injected `<|tool_result|>…<|tool_result_end|>` and speak a natural answer
4. **Re-call** the tool on follow-up questions instead of parroting a stale answer
5. **Stay quiet / not call** during silence and ordinary conversation

Design notes:
- **Call/result formats match runtime exactly** — call is `get_weather London`
  (plain city), result is `London: Clear, +27°C` (wttr.in `%C, %t`) and time is
  `8:31 PM on Friday, June 19, 2026` (`get_time()` output).
- **Masking is principled**: the model is trained (mask=1) only on the tokens it
  must *emit* — the tool call and the spoken response. The injected result block
  is context (mask=0) because at runtime it is force-fed, not predicted.
- **Heavy negatives**: silence, chit-chat, and distractor sentences that contain
  the words "time"/"weather" but must NOT trigger a tool.

Output saved to `data/tool_calls.jsonl`.

In [ ]:
# Install into the EXACT python running this kernel (most reliable in Modal)
import sys
!{sys.executable} -m pip install sentencepiece huggingface_hub einops sphn numpy -q
import importlib
for m in ["sentencepiece", "einops", "sphn", "huggingface_hub"]:
    importlib.import_module(m)
print("All deps importable ✓")

In [ ]:
import json, random, sys, os
from pathlib import Path

# ── Config — fill these in ────────────────────────────────────────────────────
HF_PATCHED_REPO = "abrarfahim/moshi-tool-patched"
HF_TOKEN        = os.environ.get("HF_TOKEN", None)  # or paste token string here
# ─────────────────────────────────────────────────────────────────────────────

# Clone repo if not present (Modal environment)
REPO = Path("/repo")
if not REPO.exists():
    import subprocess
    subprocess.run(["git", "clone",
        "https://github.com/syedfahimabrar/moshi-D-gu.git",
        str(REPO)], check=True)

sys.path.insert(0, str(REPO / "moshi"))

import sentencepiece as spm

TOKENIZER_PATH = REPO / "weights/patched/tokenizer_spm_32k_3.model"
if not TOKENIZER_PATH.exists():
    from huggingface_hub import hf_hub_download
    TOKENIZER_PATH = Path(hf_hub_download(
        repo_id=HF_PATCHED_REPO, filename="tokenizer_spm_32k_3.model",
        local_dir="/tmp/patched", token=HF_TOKEN))

tok = spm.SentencePieceProcessor()
tok.Load(str(TOKENIZER_PATH))
print(f"Tokenizer: {tok.get_piece_size()} tokens")

In [ ]:
PAD_ID             = 3
TOOL_CALL_ID       = 32000
TOOL_END_ID        = 32001
TOOL_RESULT_ID     = 32002
TOOL_RESULT_END_ID = 32003

assert tok.id_to_piece(TOOL_CALL_ID) == "<|tool_call|>", "Tokenizer not patched!"
print("Special tokens:", [tok.id_to_piece(i) for i in [32000,32001,32002,32003]])

In [ ]:
# All generators live in the shared module gen_tool_data.py so notebooks 01 and
# 02 never drift. (Notebook 02 calls this same module inline after cloning.)
sys.path.insert(0, str(REPO / "notebooks"))
import gen_tool_data
from gen_tool_data import (
    PAD_ID, TOOL_CALL_ID, TOOL_END_ID, TOOL_RESULT_ID, TOOL_RESULT_END_ID,
)
g = gen_tool_data._make(tok)   # generator functions bound to this tokenizer

print("time:   ", tok.decode([t for t in g["time_example"]()["tokens"] if t < 32000]))
print("weather:", tok.decode([t for t in g["weather_example"]("London")["tokens"] if t < 32000]))

In [ ]:
# Generate the full dataset (~2300 examples)
data = gen_tool_data.generate(tok)

from collections import Counter
counts = Counter(e["type"] for e in data)
n_pos = sum(v for k, v in counts.items() if k.startswith(("time", "weather")))
n_neg = len(data) - n_pos
print(f"Total: {len(data)} examples  ({n_pos} positive / {n_neg} negative)")
print("Breakdown:", {k: counts[k] for k in sorted(counts)})
print(f"Avg length: {sum(len(e['tokens']) for e in data)/len(data):.0f} tokens")
print(f"Max length: {max(len(e['tokens']) for e in data)} tokens")

In [ ]:
# Save to data/tool_calls.jsonl
OUT = REPO / "notebooks/data/tool_calls.jsonl"
import json
OUT.parent.mkdir(parents=True, exist_ok=True)
with OUT.open("w") as f:
    for ex in data:
        f.write(json.dumps(ex) + "\n")
print(f"Saved {len(data)} examples → {OUT}")
print("\nNote: notebook 02 regenerates this inline via gen_tool_data, so you do")
print("NOT need to commit this file for training — it's only for inspection.")

In [ ]:
# Preview one of each type, marking trained tokens (mask=1 UPPER, mask=0 lower)
def render(ex):
    out = []
    for tid, m in zip(ex["tokens"], ex["mask"]):
        if   tid == TOOL_CALL_ID:       s = "‹CALL›"
        elif tid == TOOL_END_ID:        s = "‹END›"
        elif tid == TOOL_RESULT_ID:     s = "‹RES›"
        elif tid == TOOL_RESULT_END_ID: s = "‹/RES›"
        elif tid == PAD_ID:             s = "·"
        else:                           s = tok.id_to_piece(tid).replace("▁", " ")
        out.append(s if m else (s.lower() if s.isalpha() else s))
    return "".join(out)

for t in ["time", "time_multiturn", "weather", "weather_local",
          "weather_multiturn", "silence", "chitchat", "distractor"]:
    ex = next((e for e in data if e["type"] == t), None)
    if ex:
        print(f"[{t}]")
        print("  ", render(ex)[:170])
        print(f"   trained tokens (mask=1): {sum(ex['mask'])}/{len(ex['mask'])}\n")